# Data Cleaning Project
This notebook cleans the dataset step-by-step. It has been specifically updated to clean the 'status', 'card_type', 'city', and 'amount' columns.

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

# 1. Load the dataset
file_name = 'trx-10k.csv.xlsx'

try:
    df = pd.read_excel(file_name)
    print("Dataset loaded successfully!")
except Exception as e:
    print(f"Error loading file: {e}")
    df = pd.DataFrame({'id': ['1', '2'], 'status': ['success', 'fail'], 'amount': [100.0, -999999.0]})


## Data Quality Report (Before Cleaning)
Let's check the current state of our data: nulls, duplicates, and data types.

In [ ]:
# Store 'before' metrics for our final comparison table
rows_before = len(df)
nulls_before = df.isnull().sum().sum()
duplicates_before = df.duplicated().sum()

print("--- DATA QUALITY REPORT ---")
print(f"Total Rows: {rows_before}")
print("\n1. Null values per column:")
print(df.isnull().sum())
print(f"\n2. Duplicate rows: {duplicates_before}")
print("\n3. Data types:")
print(df.dtypes)


## 1. Duplicate Removal
We will remove exact duplicate rows from the dataset.

In [ ]:
# Drop duplicates
df = df.drop_duplicates()
print(f"Removed {duplicates_before - df.duplicated().sum()} duplicate rows. New row count: {len(df)}")


## 2. Standardisation (Targeted Fixes)
Fixing inconsistent text formatting in 'status', 'card_type', and 'city'.

In [ ]:
# Fix 'status' column
if 'status' in df.columns:
    df['status'] = df['status'].astype(str).str.lower().str.strip()
    df['status'] = df['status'].replace({'succeed': 'success', 'failed': 'fail'})
    df['status'] = df['status'].str.title()
    print("Standardised 'status' column.")

# Fix 'card_type' column
if 'card_type' in df.columns:
    df['card_type'] = df['card_type'].astype(str).str.lower().str.strip()
    df['card_type'] = df['card_type'].replace({'mastcard': 'mastercard', 'master-card': 'mastercard', 'amex': 'american express', 'vsa': 'visa'})
    df['card_type'] = df['card_type'].str.title()
    print("Standardised 'card_type' column.")

# Fix 'city' column
if 'city' in df.columns:
    df['city'] = df['city'].astype(str).str.lower().str.strip()
    df['city'] = df['city'].replace({'thr': 'tehran', 'tehr@n': 'tehran', 'thran': 'tehran', 'nan': 'unknown'})
    df['city'] = df['city'].str.title()
    print("Standardised 'city' column.")


## 3. Data Type Correction
Ensuring dates are `datetime` objects, IDs are strings, and monetary values are floats.

In [ ]:
# 1. Convert 'time' column to datetime
if 'time' in df.columns:
    df['time'] = pd.to_datetime(df['time'], errors='coerce')
    print("Converted 'time' to datetime.")

# 2. Convert 'id' to string (remove the .0 from floats)
if 'id' in df.columns:
    df['id'] = df['id'].astype(str).str.replace(r'\.0$', '', regex=True)
    print("Converted 'id' to string.")

# 3. Convert 'amount' to float and replace -999999.0 with NaN (missing value)
if 'amount' in df.columns:
    df['amount'] = pd.to_numeric(df['amount'], errors='coerce')
    # Replacing negative values (like -999999.0) with NaN so we can fill them with median later
    df.loc[df['amount'] < 0, 'amount'] = np.nan
    print("Converted 'amount' to float and turned negative placeholders into NaN.")


## 4. Missing Data Handling
Decisions for missing data:
- **Categorical columns (like Gender)**: We will fill with the mode (most common value).
- **Numeric columns (like Age or Salary)**: We will fill with the median, because median is resistant to extreme outliers.

In [ ]:
# Fill numeric columns with median
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
for col in numeric_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

# Fill categorical/text columns with Mode
text_cols = df.select_dtypes(include=['object']).columns
for col in text_cols:
    if col != 'id': # Don't fill missing IDs with mode
        mode_val = df[col].mode()[0] if not df[col].mode().empty else 'Unknown'
        df[col] = df[col].fillna(mode_val)

print("Missing values handled!")


## 5. Outlier Detection (Numeric Columns)
We will use the **IQR (Interquartile Range) method** to find and cap outliers.

In [ ]:
def cap_outliers_iqr(dataframe, column):
    Q1 = dataframe[column].quantile(0.25)
    Q3 = dataframe[column].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Count how many outliers
    outliers_count = ((dataframe[column] < lower_bound) | (dataframe[column] > upper_bound)).sum()
    
    # Cap the values
    dataframe[column] = np.where(dataframe[column] < lower_bound, lower_bound, dataframe[column])
    dataframe[column] = np.where(dataframe[column] > upper_bound, upper_bound, dataframe[column])
    
    if outliers_count > 0:
        print(f"{column}: Capped {outliers_count} outliers.")

for col in numeric_cols:
    cap_outliers_iqr(df, col)


## Before vs. After Summary
Let's compare our dataset's quality before and after the cleaning process.

In [ ]:
rows_after = len(df)
nulls_after = df.isnull().sum().sum()
duplicates_after = df.duplicated().sum()

summary_data = {
    'Metric': ['Total Rows', 'Total Null Values', 'Duplicate Rows'],
    'Before Cleaning': [rows_before, nulls_before, duplicates_before],
    'After Cleaning': [rows_after, nulls_after, duplicates_after]
}

summary_df = pd.DataFrame(summary_data)
print("\n--- CLEANING SUMMARY ---")
display(summary_df)


## Save the Cleaned Dataset
Finally, we save the clean data to a new CSV file so it's ready for analysis!

In [ ]:
clean_file_name = "cleaned_dataset.csv"
df.to_csv(clean_file_name, index=False)
print(f"Success! Cleaned data saved as '{clean_file_name}'")
